# Chapter 0: Prepare OSM Data

This notebook downloads and clips an OSM road network file for cities that use a pre-clipped local file rather than the Overpass API.

**If you cloned the GitHub repository, the clipped files for San Francisco (`sf-central-southeast.osm.pbf`) and Singapore (`singapore-central.osm.pbf`) are already included. You do not need to run this notebook -- go straight to `02_road_network.ipynb`.**

Run this notebook only if you want to:
- Regenerate the clipped files from the latest Geofabrik data
- Create a clipped file for your own city config

The default Merton config does not need this notebook.

## Steps

1. Copy your chosen city config to `config.yaml`
2. Run this notebook
3. Run `02_road_network.ipynb` as normal

## 1. Install Dependencies

In [1]:
%pip install networkx==3.6.1 \
             pyrosm==0.13.1 \
             pyyaml==6.0.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import networkx as nx
import os
import urllib.request

from config_validator import load_config, ConfigError
from pyrosm import OSM

## 3. Load Configuration

In [3]:
try:
    cfg = load_config("config.yaml")
except (FileNotFoundError, ConfigError) as e:
    raise SystemExit(f"Config error: {e}")

city = cfg["city"]

SKIP = "osm_file" not in city or not city["osm_file"]

if SKIP:
    print(f"City '{city['name']}' does not use a local OSM file.")
    print(f"Run 02_road_network.ipynb directly.")
else:
    OSM_SOURCE_FILE = city["osm_source_file"]
    OSM_SOURCE_URL  = city["osm_source"]
    OSM_OUTPUT_FILE = city["osm_file"]
    OSM_BBOX        = city["osm_bbox"]  # [south, north, west, east]
    south, north, west, east = OSM_BBOX
    print(f"City        : {city['name']}")
    print(f"Source file : {OSM_SOURCE_FILE}")
    print(f"Output file : {OSM_OUTPUT_FILE}")
    print(f"Bbox        : south={south}, north={north}, west={west}, east={east}")

City 'London Borough of Merton' does not use a local OSM file.
Run 02_road_network.ipynb directly.


## 4. Download Source OSM File

Download only happens if the file is not already present.

In [4]:
if not SKIP:
    if os.path.exists(OSM_SOURCE_FILE):
        size_mb = os.path.getsize(OSM_SOURCE_FILE) / (1024 * 1024)
        print(f"Source file already exists: {OSM_SOURCE_FILE} ({size_mb:.0f} MB) -- skipping download.")
    else:
        print(f"Downloading {OSM_SOURCE_FILE} from Geofabrik...")
        print(f"URL: {OSM_SOURCE_URL}")
        print("This may take several minutes.")

        def _progress(count, block_size, total_size):
            pct = min(100, count * block_size * 100 // total_size)
            mb  = count * block_size / (1024 * 1024)
            print(f"\r  {pct}% ({mb:.0f} MB)", end="", flush=True)

        urllib.request.urlretrieve(OSM_SOURCE_URL, OSM_SOURCE_FILE, reporthook=_progress)
        size_mb = os.path.getsize(OSM_SOURCE_FILE) / (1024 * 1024)
        print(f"\nDownloaded: {OSM_SOURCE_FILE} ({size_mb:.0f} MB)")

## 5. Clip to Bounding Box

In [5]:
if not SKIP:
    if os.path.exists(OSM_OUTPUT_FILE):
        size_mb = os.path.getsize(OSM_OUTPUT_FILE) / (1024 * 1024)
        print(f"Clipped file already exists: {OSM_OUTPUT_FILE} ({size_mb:.1f} MB) -- skipping clip.")
        print(f"Delete {OSM_OUTPUT_FILE} and re-run to regenerate.")
    else:
        print(f"Clipping {OSM_SOURCE_FILE} to bbox...")

        # pyrosm bbox order: [west, south, east, north]
        osm = OSM(
            OSM_SOURCE_FILE,
            bounding_box=[west, south, east, north]
        )

        clipped_path = osm.to_pbf(OSM_OUTPUT_FILE)
        size_mb = os.path.getsize(clipped_path) / (1024 * 1024)
        print(f"Saved: {clipped_path} ({size_mb:.1f} MB)")

## 6. Verify Clipped File

In [6]:
if not SKIP:
    osm_check = OSM(OSM_OUTPUT_FILE)
    nodes, edges = osm_check.get_network(network_type="driving", nodes=True)
    G = osm_check.to_graph(nodes, edges, graph_type="networkx")

    components = list(nx.strongly_connected_components(G))
    largest    = max(len(c) for c in components)

    print(f"Nodes                         : {len(G.nodes):,}")
    print(f"Edges                         : {len(G.edges):,}")
    print(f"Strongly connected components : {len(components)}")
    print(f"Largest component             : {largest:,} nodes")
    print()
    print(f"Ready to run 02_road_network.ipynb with config: {city['name']}")